# Plantilla Colab – Subtask 2 (línea TartuNLP)

Copia este notebook para cada experimento nuevo. **La lógica va en el repo** (`semeval27/` y `scripts/`), aquí solo se prepara el entorno y se llaman los scripts.

Flujo: cambias el código en VS Code → `git push` → aquí ejecutas las celdas 1–3 (hacen `git pull`) → experimento.

Datos y resultados viven en **Drive** (`MyDrive/semeval27/...`), nunca en el repo.

## 1. Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
DATA_DIR = f'{DRIVE}/semeval27/dev'                       # SV/, ... con usages.jsonl
ADAPTER  = f'{DRIVE}/axolotl_models/ru/classification'    # adaptador TartuNLP-ru
PREDS    = f'{DRIVE}/semeval27/preds'
RESULTS  = f'{DRIVE}/semeval27/results'
!mkdir -p {PREDS} {RESULTS}

## 2. Repo (clonar la primera vez, `pull` las siguientes)
Si el repo es privado: guarda un token de GitHub en los *Secrets* de Colab (icono de la llave) con el nombre `GITHUB_TOKEN`.

In [ ]:
import os
REPO_DIR = '/content/repo'
url = 'https://github.com/tonyOehme/hcc_practical_course_semeval2027.git'
try:
    from google.colab import userdata
    url = url.replace('https://', f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass  # repo público o sin token

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}/task2/semeval27_pipeline
!git log --oneline -1

## 3. Entorno
Crea `/content/venv` (Python 3.11 + versiones de TartuNLP) e instala el paquete `semeval27`. La primera vez tarda unos minutos; después es instantáneo.
Para análisis solo en CPU basta con `--core`.

In [ ]:
!bash scripts/colab_setup.sh
PY = '/content/venv/bin/python'

## 4. Experimento

In [ ]:
# Ejemplo: pipeline completo con el adaptador ruso
!{PY} scripts/convert_data.py --data_dir {DATA_DIR} --out_dir outputs/pairs
!{PY} scripts/predict_tartunlp.py --pairs outputs/pairs/SV.pairs.tsv --adapter_dir {ADAPTER} --out {PREDS}/SV.tartunlp_ru.tsv
!{PY} scripts/evaluate_subtask2.py --pred {PREDS}/SV.tartunlp_ru.tsv --threshold 0.35 --out_dir {RESULTS}